In [ ]:
# Install required libraries
!pip install langchain langchain-core langchain-community --quiet

print("All packages installed!")

 All packages installed!


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.9 requires langchain-core>=1.0.0, but you have langchain-core 0.3.84 which is incompatible.


In [2]:
# ─── Imports ────────────────────────────────────────────────────────────────────
import json
import os

from langchain.prompts import (
    PromptTemplate,
    FewShotPromptTemplate,
    ChatPromptTemplate,
    PipelinePromptTemplate,
)
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

print("All imports ready!")

 All imports ready!


In [3]:
# ─── Task 1: Few-Shot Prompt Template ───────────────────────────────────────────

# Step 1 — Define shot examples (no hardcoding in final prompt)
FEW_SHOT_EXAMPLES = [
    {
        "topic": "Photosynthesis",
        "explanation": (
            "Photosynthesis is like a kitchen in a plant. "
            "It takes sunlight, water, and air to cook food (glucose) for itself."
        )
    },
    {
        "topic": "Gravity",
        "explanation": (
            "Gravity is an invisible force that pulls everything toward the ground. "
            "It is why a dropped ball falls instead of floating away."
        )
    },
    {
        "topic": "DNA",
        "explanation": (
            "DNA is like a recipe book inside every cell. "
            "It holds all the instructions needed to build and run a living body."
        )
    },
]

# Step 2 — Template for formatting each individual example
example_formatter = PromptTemplate(
    input_variables=["topic", "explanation"],
    template="Topic: {topic}\nExplanation: {explanation}"
)

# Step 3 — Build the FewShotPromptTemplate
few_shot_template = FewShotPromptTemplate(
    examples=FEW_SHOT_EXAMPLES,
    example_prompt=example_formatter,
    prefix="You explain complex topics using simple analogies. Here are some examples:\n",
    suffix="\nNow explain the following:\nTopic: {topic}\nExplanation:",
    input_variables=["topic"],
    example_separator="\n---\n"
)

# Step 4 — Generate prompts for new topics
new_topics = ["Machine Learning", "Encryption", "Quantum Computing"]

print("=" * 60)
print("Task 1: Few-Shot Prompt Template")
print("=" * 60)

for topic in new_topics:
    prompt = few_shot_template.format(topic=topic)
    print(f"\nGenerated Prompt for: '{topic}'")
    print("-" * 50)
    print(prompt)
    print()

print("Task 1 Complete — FewShotPromptTemplate working!")

Task 1: Few-Shot Prompt Template

 Generated Prompt for: 'Machine Learning'
--------------------------------------------------
You explain complex topics using simple analogies. Here are some examples:

---
Topic: Photosynthesis
Explanation: Photosynthesis is like a kitchen in a plant. It takes sunlight, water, and air to cook food (glucose) for itself.
---
Topic: Gravity
Explanation: Gravity is an invisible force that pulls everything toward the ground. It is why a dropped ball falls instead of floating away.
---
Topic: DNA
Explanation: DNA is like a recipe book inside every cell. It holds all the instructions needed to build and run a living body.
---

Now explain the following:
Topic: Machine Learning
Explanation:


 Generated Prompt for: 'Encryption'
--------------------------------------------------
You explain complex topics using simple analogies. Here are some examples:

---
Topic: Photosynthesis
Explanation: Photosynthesis is like a kitchen in a plant. It takes sunlight, water

In [ ]:
# ─── Task 2: Partial Prompt Templates ───────────────────────────────────────────

# Full template with 4 variables
full_code_template = PromptTemplate(
    input_variables=["language", "difficulty", "concept", "output_format"],
    template=(
        "Write a {difficulty}-level {language} code example that demonstrates {concept}. "
        "Format the response as {output_format}."
    )
)

# ── Partial 1: Fix language + output_format → only need difficulty + concept ──
python_template = full_code_template.partial(
    language="Python",
    output_format="a commented code block with explanation"
)

# ── Partial 2: Fix language + difficulty → only need concept + output_format ──
beginner_js_template = full_code_template.partial(
    language="JavaScript",
    difficulty="beginner"
)

# ── Partial 3: Fix all but concept ──────────────────────────────────────────────
ready_template = full_code_template.partial(
    language="SQL",
    difficulty="intermediate",
    output_format="a table of examples"
)

print("=" * 60)
print("Task 2: Partial Prompt Templates")
print("=" * 60)

# Use Partial 1 — still needs difficulty + concept
print("\nPartial 1: Python template (needs difficulty + concept)")
p1 = python_template.format(difficulty="advanced", concept="decorators")
print("  ", p1)

# Use Partial 2 — still needs concept + output_format
print("\nPartial 2: Beginner JS template (needs concept + output_format)")
p2 = beginner_js_template.format(concept="closures", output_format="step-by-step explanation")
print("  ", p2)

# Use Partial 3 — only needs concept
print("\nPartial 3: SQL template (needs only concept)")
concepts = ["JOINs", "GROUP BY", "window functions"]
for c in concepts:
    print(f"   • {ready_template.format(concept=c)}")

print("\nTask 2 Complete — Partial templates working!")

Task 2: Partial Prompt Templates

 Partial 1: Python template (needs difficulty + concept)
   Write a advanced-level Python code example that demonstrates decorators. Format the response as a commented code block with explanation.

 Partial 2: Beginner JS template (needs concept + output_format)
   Write a beginner-level JavaScript code example that demonstrates closures. Format the response as step-by-step explanation.

 Partial 3: SQL template (needs only concept)
   • Write a intermediate-level SQL code example that demonstrates JOINs. Format the response as a table of examples.
   • Write a intermediate-level SQL code example that demonstrates GROUP BY. Format the response as a table of examples.
   • Write a intermediate-level SQL code example that demonstrates window functions. Format the response as a table of examples.

 Task 2 Complete — Partial templates working!


In [ ]:
# ─── Task 3: Pipeline Prompt Template ───────────────────────────────────────────

# ── Sub-template 1: Persona intro ───────────────────────────────────────────────
intro_template = PromptTemplate(
    input_variables=["role", "expertise"],
    template="You are a {role} with deep expertise in {expertise}."
)

# ── Sub-template 2: Communication style ─────────────────────────────────────────
style_template = PromptTemplate(
    input_variables=["tone", "audience"],
    template="Always communicate in a {tone} tone, suitable for {audience}."
)

# ── Sub-template 3: Specific task ───────────────────────────────────────────────
task_template = PromptTemplate(
    input_variables=["action", "topic"],
    template="Your task: {action} about {topic}. Be thorough but concise."
)

# ── Full final template that references rendered sub-templates ───────────────────
final_template = PromptTemplate(
    input_variables=["intro", "style", "task"],
    template="{intro}\n\n{style}\n\n{task}"
)

# ── Assemble into a Pipeline ─────────────────────────────────────────────────────
pipeline_prompt = PipelinePromptTemplate(
    final_prompt=final_template,
    pipeline_prompts=[
        ("intro", intro_template),
        ("style", style_template),
        ("task",  task_template),
    ]
)

# ── Test Scenarios ───────────────────────────────────────────────────────────────
scenarios = [
    {
        "role": "data science educator",
        "expertise": "machine learning",
        "tone": "friendly and encouraging",
        "audience": "college students",
        "action": "write a beginner tutorial",
        "topic": "gradient descent"
    },
    {
        "role": "senior software architect",
        "expertise": "distributed systems",
        "tone": "precise and technical",
        "audience": "senior engineers",
        "action": "write a design document",
        "topic": "event-driven microservices"
    },
]

print("=" * 60)
print("Task 3: Pipeline Prompt Template")
print("=" * 60)

for i, scenario in enumerate(scenarios, 1):
    result = pipeline_prompt.format(**scenario)
    print(f"\nScenario {i}: {scenario['role'].upper()}")
    print("-" * 50)
    print(result)
    print()

print("Task 3 Complete — PipelinePromptTemplate working!")

Task 3: Pipeline Prompt Template

 Scenario 1: DATA SCIENCE EDUCATOR
--------------------------------------------------
You are a data science educator with deep expertise in machine learning.

Always communicate in a friendly and encouraging tone, suitable for college students.

Your task: write a beginner tutorial about gradient descent. Be thorough but concise.


 Scenario 2: SENIOR SOFTWARE ARCHITECT
--------------------------------------------------
You are a senior software architect with deep expertise in distributed systems.

Always communicate in a precise and technical tone, suitable for senior engineers.

Your task: write a design document about event-driven microservices. Be thorough but concise.

 Task 3 Complete — PipelinePromptTemplate working!


C:\Users\omdon\AppData\Local\Temp\ipykernel_22644\2234831033.py:28: LangChainDeprecationWarning: This class is deprecated in favor of chaining individual prompts together.
  pipeline_prompt = PipelinePromptTemplate(


In [ ]:
# ─── Task 4: Conditional Prompt Routing ─────────────────────────────────────────

# ── Define a bank of conditional templates ──────────────────────────────────────
CONDITIONAL_TEMPLATES = {
    ("beginner", "understand"): PromptTemplate(
        input_variables=["topic"],
        template=(
            "Explain {topic} using a simple real-life analogy. "
            "Avoid jargon. Use short sentences."
        )
    ),
    ("beginner", "implement"): PromptTemplate(
        input_variables=["topic"],
        template=(
            "Give a beginner-friendly step-by-step guide to implement {topic}. "
            "Include a minimal working example."
        )
    ),
    ("intermediate", "understand"): PromptTemplate(
        input_variables=["topic"],
        template=(
            "Explain the mechanics and intuition behind {topic}. "
            "Connect concepts to real-world applications."
        )
    ),
    ("intermediate", "implement"): PromptTemplate(
        input_variables=["topic"],
        template=(
            "Provide a practical implementation guide for {topic} with "
            "common patterns, pitfalls, and best practices."
        )
    ),
    ("expert", "understand"): PromptTemplate(
        input_variables=["topic"],
        template=(
            "Give a deep theoretical analysis of {topic} including "
            "mathematical foundations, limitations, and current research frontiers."
        )
    ),
    ("expert", "implement"): PromptTemplate(
        input_variables=["topic"],
        template=(
            "Provide an advanced implementation blueprint for {topic} covering "
            "architecture decisions, optimization strategies, and production concerns."
        )
    ),
}

# ── Router function ──────────────────────────────────────────────────────────────
def route_prompt(topic: str, level: str, goal: str) -> str:
    """
    Routes to the correct PromptTemplate based on (level, goal).

    Args:
        topic : Subject to explain
        level : 'beginner' | 'intermediate' | 'expert'
        goal  : 'understand' | 'implement'

    Returns:
        Formatted prompt string
    """
    key = (level.lower(), goal.lower())
    if key not in CONDITIONAL_TEMPLATES:
        raise KeyError(
            f"No template for (level='{level}', goal='{goal}'). "
            f"Valid options: {list(CONDITIONAL_TEMPLATES.keys())}"
        )
    return CONDITIONAL_TEMPLATES[key].format(topic=topic)


# ── Test all routes ──────────────────────────────────────────────────────────────
routing_tests = [
    {"topic": "Transformers",    "level": "beginner",     "goal": "understand"},
    {"topic": "Transformers",    "level": "beginner",     "goal": "implement"},
    {"topic": "Backpropagation", "level": "intermediate", "goal": "understand"},
    {"topic": "GANs",            "level": "intermediate", "goal": "implement"},
    {"topic": "RLHF",            "level": "expert",       "goal": "understand"},
    {"topic": "RLHF",            "level": "expert",       "goal": "implement"},
]

print("=" * 60)
print("Task 4: Conditional Prompt Routing")
print("=" * 60)

for test in routing_tests:
    prompt = route_prompt(**test)
    print(f"\n[{test['level'].upper()} | {test['goal'].upper()}] → Topic: '{test['topic']}'")
    print(f"   {prompt}")

print("\nTask 4 Complete — Conditional routing working!")

Task 4: Conditional Prompt Routing

 [BEGINNER | UNDERSTAND] → Topic: 'Transformers'
   Explain Transformers using a simple real-life analogy. Avoid jargon. Use short sentences.

 [BEGINNER | IMPLEMENT] → Topic: 'Transformers'
   Give a beginner-friendly step-by-step guide to implement Transformers. Include a minimal working example.

 [INTERMEDIATE | UNDERSTAND] → Topic: 'Backpropagation'
   Explain the mechanics and intuition behind Backpropagation. Connect concepts to real-world applications.

 [INTERMEDIATE | IMPLEMENT] → Topic: 'GANs'
   Provide a practical implementation guide for GANs with common patterns, pitfalls, and best practices.

 [EXPERT | UNDERSTAND] → Topic: 'RLHF'
   Give a deep theoretical analysis of RLHF including mathematical foundations, limitations, and current research frontiers.

 [EXPERT | IMPLEMENT] → Topic: 'RLHF'
   Provide an advanced implementation blueprint for RLHF covering architecture decisions, optimization strategies, and production concerns.

 Tas

In [ ]:
# ─── Task 5: Prompt Serialization ───────────────────────────────────────────────

import os
from langchain.prompts import load_prompt

# ── Templates to serialize ───────────────────────────────────────────────────────
SERIALIZABLE_TEMPLATES = {
    "tutor_template": PromptTemplate(
        input_variables=["topic", "audience"],
        template="Teach {topic} to a {audience}. Use clear examples and check for understanding."
    ),
    "quiz_template": PromptTemplate(
        input_variables=["topic", "num_questions"],
        template="Create {num_questions} multiple-choice quiz questions about {topic}. Include answer keys."
    ),
    "summary_template": PromptTemplate(
        input_variables=["topic", "length"],
        template="Summarize {topic} in {length} sentences. Focus on key ideas only."
    ),
}

SAVE_DIR = "prompt_templates"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Save all templates ────────────────────────────────────────────────────────────
print("=" * 60)
print("Task 5: Prompt Serialization")
print("=" * 60)

print("\nSaving templates...")
saved_paths = {}
for name, template in SERIALIZABLE_TEMPLATES.items():
    path = os.path.join(SAVE_DIR, f"{name}.json")
    template.save(path)
    saved_paths[name] = path
    print(f"   Saved: {path}")

# ── Inspect saved JSON ────────────────────────────────────────────────────────────
print("\nInspecting saved JSON (tutor_template):")
with open(saved_paths["tutor_template"], "r") as f:
    data = json.load(f)
print(json.dumps(data, indent=2))

# ── Reload templates ──────────────────────────────────────────────────────────────
print("\nReloading and using templates...")
for name, path in saved_paths.items():
    loaded = load_prompt(path)
    print(f"\nReloaded: '{name}'")
    print(f"   input_variables: {loaded.input_variables}")
    print(f"   template       : {loaded.template}")

# ── Test reloaded template ────────────────────────────────────────────────────────
print("\nUsing reloaded quiz_template:")
quiz = load_prompt(saved_paths["quiz_template"])
print(" ", quiz.format(topic="Python Decorators", num_questions="5"))

print("\nTask 5 Complete — Templates saved and reloaded!")

Task 5: Prompt Serialization

 Saving templates...
    Saved: prompt_templates\tutor_template.json
    Saved: prompt_templates\quiz_template.json
    Saved: prompt_templates\summary_template.json

 Inspecting saved JSON (tutor_template):
{
  "name": null,
  "input_variables": [
    "audience",
    "topic"
  ],
  "optional_variables": [],
  "output_parser": null,
  "partial_variables": {},
  "metadata": null,
  "tags": null,
  "template": "Teach {topic} to a {audience}. Use clear examples and check for understanding.",
  "template_format": "f-string",
  "validate_template": false,
  "_type": "prompt"
}

 Reloading and using templates...

 Reloaded: 'tutor_template'
   input_variables: ['audience', 'topic']
   template       : Teach {topic} to a {audience}. Use clear examples and check for understanding.

 Reloaded: 'quiz_template'
   input_variables: ['num_questions', 'topic']
   template       : Create {num_questions} multiple-choice quiz questions about {topic}. Include answer keys.



In [ ]:
# ─── Task 6: AI Tutor Prompt Engine ─────────────────────────────────────────────

# ── Constants ────────────────────────────────────────────────────────────────────
VALID_LEVELS   = ["beginner", "intermediate", "advanced"]
VALID_SESSIONS = ["explain", "quiz", "summarize", "practice"]

# ── Session Templates ─────────────────────────────────────────────────────────────
SESSION_TEMPLATES = {
    "explain": PromptTemplate(
        input_variables=["name", "level", "topic"],
        template=(
            "You are tutoring {name}, a {level} student. "
            "Explain {topic} with appropriate depth, clear examples, "
            "and end with a comprehension check question."
        )
    ),
    "quiz": PromptTemplate(
        input_variables=["name", "level", "topic"],
        template=(
            "Generate a 5-question {level}-level quiz for {name} on {topic}. "
            "Mix question types: MCQ, true/false, and one short answer. "
            "Provide answer keys at the end."
        )
    ),
    "summarize": PromptTemplate(
        input_variables=["name", "level", "topic"],
        template=(
            "Create a concise {level}-appropriate summary of {topic} for {name}. "
            "Use bullet points for key takeaways and a one-paragraph conclusion."
        )
    ),
    "practice": PromptTemplate(
        input_variables=["name", "level", "topic"],
        template=(
            "Design a hands-on practice exercise about {topic} for {name} at {level} level. "
            "Include: objective, steps, expected output, and a challenge extension."
        )
    ),
}

# ── Validation ────────────────────────────────────────────────────────────────────
def validate_tutor_inputs(level: str, session: str) -> tuple:
    """
    Validates level and session type for the AI Tutor engine.
    Returns cleaned and validated (level, session) tuple.
    """
    level_clean   = level.strip().lower()
    session_clean = session.strip().lower()

    if level_clean not in VALID_LEVELS:
        raise ValueError(f"Invalid level '{level}'. Choose from: {VALID_LEVELS}")
    if session_clean not in VALID_SESSIONS:
        raise ValueError(f"Invalid session '{session}'. Choose from: {VALID_SESSIONS}")

    return level_clean, session_clean


# ── Main Engine Function ─────────────────────────────────────────────────────────
def ai_tutor_prompt(name: str, level: str, topic: str, session: str) -> str:
    """
    Generates a structured tutor prompt for a student.

    Args:
        name    : Student's name
        level   : 'beginner' | 'intermediate' | 'advanced'
        topic   : Subject/topic to cover
        session : 'explain' | 'quiz' | 'summarize' | 'practice'

    Returns:
        Structured prompt string
    """
    v_level, v_session = validate_tutor_inputs(level, session)
    template = SESSION_TEMPLATES[v_session]
    return template.format(name=name, level=v_level, topic=topic)


# ── Student Profiles ──────────────────────────────────────────────────────────────
student_sessions = [
    {"name": "Arjun",  "level": "beginner",      "topic": "Neural Networks",       "session": "explain"},
    {"name": "Priya",  "level": "intermediate",   "topic": "Convolutional Networks", "session": "quiz"},
    {"name": "Rahul",  "level": "advanced",       "topic": "Attention Mechanisms",   "session": "practice"},
    {"name": "Sneha",  "level": "beginner",       "topic": "Python Functions",       "session": "summarize"},
    {"name": "Vikram", "level": "intermediate",   "topic": "REST APIs",              "session": "explain"},
]

print("=" * 65)
print("Task 6: AI Tutor Prompt Engine")
print("=" * 65)

for student in student_sessions:
    prompt = ai_tutor_prompt(**student)
    print(f"\nStudent: {student['name']} | Level: {student['level']} | Session: {student['session'].upper()}")
    print(f"   Topic  : {student['topic']}")
    print(f"   Prompt : {prompt}")

print("\nTask 6 Complete — AI Tutor Engine working!")

Task 6: AI Tutor Prompt Engine

 Student: Arjun | Level: beginner | Session: EXPLAIN
   Topic  : Neural Networks
   Prompt : You are tutoring Arjun, a beginner student. Explain Neural Networks with appropriate depth, clear examples, and end with a comprehension check question.

 Student: Priya | Level: intermediate | Session: QUIZ
   Topic  : Convolutional Networks
   Prompt : Generate a 5-question intermediate-level quiz for Priya on Convolutional Networks. Mix question types: MCQ, true/false, and one short answer. Provide answer keys at the end.

 Student: Rahul | Level: advanced | Session: PRACTICE
   Topic  : Attention Mechanisms
   Prompt : Design a hands-on practice exercise about Attention Mechanisms for Rahul at advanced level. Include: objective, steps, expected output, and a challenge extension.

 Student: Sneha | Level: beginner | Session: SUMMARIZE
   Topic  : Python Functions
   Prompt : Create a concise beginner-appropriate summary of Python Functions for Sneha. Use bulle

In [ ]:
# ─── Task 7: LCEL Chain with Mock LLM ───────────────────────────────────────────

from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ── Step 1: Define prompt template ───────────────────────────────────────────────
lcel_prompt = PromptTemplate(
    input_variables=["topic", "level", "style"],
    template="[{level} | {style}] Explain {topic} clearly and concisely."
)

# ── Step 2: Mock LLM (simulates a real model response) ──────────────────────────
def mock_llm(prompt_value) -> str:
    """
    Simulates an LLM call. In production, replace with:
        from langchain_openai import ChatOpenAI
        llm = ChatOpenAI(model='gpt-4o')
    """
    prompt_text = prompt_value.text if hasattr(prompt_value, 'text') else str(prompt_value)
    return f"[MOCK RESPONSE] Received prompt: '{prompt_text[:80]}...'  → (LLM would answer here)"

mock_llm_runnable = RunnableLambda(mock_llm)

# ── Step 3: Output parser ────────────────────────────────────────────────────────
output_parser = StrOutputParser()

# ── Step 4: Build LCEL Chain using pipe operator ─────────────────────────────────
#   prompt_template | mock_llm | output_parser
lcel_chain = lcel_prompt | mock_llm_runnable | output_parser

# ── Step 5: Run the chain ────────────────────────────────────────────────────────
chain_inputs = [
    {"topic": "Gradient Descent",  "level": "beginner",     "style": "visual"},
    {"topic": "Transformers",      "level": "intermediate",  "style": "technical"},
    {"topic": "RAG Architecture",  "level": "advanced",     "style": "system-design"},
]

print("=" * 65)
print("Task 7: LCEL Chain — Prompt | LLM | Output Parser")
print("=" * 65)
print("\nChain structure: lcel_prompt | mock_llm | StrOutputParser")
print("-"* 65)

for i, inputs in enumerate(chain_inputs, 1):
    result = lcel_chain.invoke(inputs)
    print(f"\nRun {i}: topic='{inputs['topic']}' | level='{inputs['level']}' | style='{inputs['style']}'")
    print(f"    {result}")

print("\n In production, replace mock_llm with:")
print("   from langchain_openai import ChatOpenAI")
print("   chain = lcel_prompt | ChatOpenAI(model='gpt-4o') | StrOutputParser()")
print("\nTask 7 Complete — LCEL chain pipeline working!")

Task 7: LCEL Chain — Prompt | LLM | Output Parser

 Chain structure: lcel_prompt | mock_llm | StrOutputParser
-----------------------------------------------------------------

  Run 1: topic='Gradient Descent' | level='beginner' | style='visual'
    [MOCK RESPONSE] Received prompt: '[beginner | visual] Explain Gradient Descent clearly and concisely....'  → (LLM would answer here)

  Run 2: topic='Transformers' | level='intermediate' | style='technical'
    [MOCK RESPONSE] Received prompt: '[intermediate | technical] Explain Transformers clearly and concisely....'  → (LLM would answer here)

  Run 3: topic='RAG Architecture' | level='advanced' | style='system-design'
    [MOCK RESPONSE] Received prompt: '[advanced | system-design] Explain RAG Architecture clearly and concisely....'  → (LLM would answer here)

 In production, replace mock_llm with:
   from langchain_openai import ChatOpenAI
   chain = lcel_prompt | ChatOpenAI(model='gpt-4o') | StrOutputParser()

 Task 7 Complete — LCEL 

In [ ]:
# ─── Bonus: Comparison Dashboard ────────────────────────────────────────────────

DEMO_TOPIC = "Reinforcement Learning"

print("=" * 65)
print(f"Prompt Template Comparison — Topic: '{DEMO_TOPIC}'")
print("=" * 65)

# 1. Basic PromptTemplate
basic = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms."
)
print("\n① Basic PromptTemplate:")
print("  ", basic.format(topic=DEMO_TOPIC))

# 2. Partial Template
partial_t = basic.partial()
advanced_partial = PromptTemplate(
    input_variables=["topic", "depth"],
    template="Give a {depth} explanation of {topic}."
).partial(depth="highly detailed mathematical")
print("\n② Partial Template (depth pre-filled):")
print("  ", advanced_partial.format(topic=DEMO_TOPIC))

# 3. Few-Shot
few_shot_demo = FewShotPromptTemplate(
    examples=[{"topic": "Supervised Learning", "answer": "Learning with labelled examples."}],
    example_prompt=PromptTemplate(
        input_variables=["topic", "answer"],
        template="Q: What is {topic}?\nA: {answer}"
    ),
    prefix="Answer the following based on the pattern below:\n",
    suffix="Q: What is {topic}?\nA:",
    input_variables=["topic"]
)
print("\n③ Few-Shot Template:")
print(few_shot_demo.format(topic=DEMO_TOPIC))

# 4. Conditional route
print("\n④ Conditional Routed Template (expert | implement):")
print(" ", route_prompt(DEMO_TOPIC, "expert", "implement"))

# 5. AI Tutor Engine
print("\n⑤ AI Tutor Engine (intermediate | quiz):")
print(" ", ai_tutor_prompt("Alex", "intermediate", DEMO_TOPIC, "quiz"))

print("\n" + "=" * 65)
print("Comparison Dashboard Complete!")
print("   Same topic → 5 different structured prompts")

  Prompt Template Comparison — Topic: 'Reinforcement Learning'

① Basic PromptTemplate:
   Explain Reinforcement Learning in simple terms.

② Partial Template (depth pre-filled):
   Give a highly detailed mathematical explanation of Reinforcement Learning.

③ Few-Shot Template:
Answer the following based on the pattern below:


Q: What is Supervised Learning?
A: Learning with labelled examples.

Q: What is Reinforcement Learning?
A:

④ Conditional Routed Template (expert | implement):
  Provide an advanced implementation blueprint for Reinforcement Learning covering architecture decisions, optimization strategies, and production concerns.

⑤ AI Tutor Engine (intermediate | quiz):
  Generate a 5-question intermediate-level quiz for Alex on Reinforcement Learning. Mix question types: MCQ, true/false, and one short answer. Provide answer keys at the end.

 Comparison Dashboard Complete!
   Same topic → 5 different structured prompts


Summary 

This set of tasks highlights key prompt engineering and pipeline design techniques in LangChain. It covers how to guide models effectively using examples, reuse prompt templates efficiently, build modular prompt pipelines, and implement conditional routing. It also emphasizes serialization for saving/loading prompts and demonstrates real-world usage through an AI tutor system. Finally, it introduces LCEL chaining, which is the modern and production-ready way to connect prompts, language models, and output parsers into a seamless pipeline.

Key Points
Few-Shot Prompting
Uses FewShotPromptTemplate to guide the model with examples before asking a question.
Partial Variables
.partial() allows you to pre-fill some variables, making templates reusable across different contexts.
Pipeline Composition
PipelinePromptTemplate helps combine multiple smaller prompts into a structured workflow.
Conditional Routing
Uses a custom router with a PromptTemplate dictionary to dynamically choose prompts based on input.
Serialization
.save() and load_prompt() enable storing and reusing prompt templates.
Real-World Application
Demonstrated through an AI Tutor multi-session engine that applies these concepts in practice.
LCEL Chain
Uses the prompt | llm | parser pipeline, which is the recommended production approach in LangChain.